# nb_mdf_50_mlv_build - Silver and Gold as materialized lake views, generated from the SAME metadata

The notebook path (`nb_mdf_20` / `nb_mdf_30`) and this MLV path read the same control tables. Here the
framework **generates declarative Spark SQL** instead of running imperative MERGE code:

```
Landing --(pipeline / nb_mdf_10)--> bronze.*  (CDF on)
                                      |
                CREATE MATERIALIZED LAKE VIEW (generated from meta.*)
                                      v
  silver_mlv.<entity>   <- column map, de-dup, SCD2, PII, DQ constraints (DROP / FAIL)
  quarantine_mlv.<entity> + quarantine_mlv.dq_rule_summary
  gold_mlv.<table>      <- the SQL already held in meta.gold_transform
```

Fabric then owns the run order (lineage), the refresh strategy (skip / incremental / full) and retries.
Every generated statement is stored in `meta.mlv_definition` so the code can be reviewed, diffed and
promoted through Azure DevOps like any other artefact.

**Prerequisites:** lakehouse with schemas enabled, Fabric Runtime 1.3, and Bronze loaded (run `nb_mdf_10`
or the orchestrator pipeline at least once).

In [ ]:
object_ids = ""              # comma-separated; empty = every active object
dry_run = False              # True = generate + store the DDL, do not execute
include_file_bronze = False  # True = also create bronze_mlv.* views that read Landing files directly (showcase)
enable_cdf = True            # turn on Change Data Feed on Bronze so optimal refresh can go incremental

In [ ]:
%run nb_mdf_utils

In [ ]:
%run nb_mdf_mlv_sqlgen

In [ ]:
def _rows(table):
    return [r.asDict() for r in spark.table(tname("meta", table)).collect()]

objects = [o for o in _rows("ingestion_object") if o.get("active")]
ids = [x.strip() for x in object_ids.split(",") if x.strip()]
if ids:
    objects = [o for o in objects if o["object_id"] in ids]
column_map = _rows("column_map")
dq_rules = _rows("dq_rule")
gold_transforms = _rows("gold_transform")

bronze_columns = {}
for o in objects:
    b = tname("bronze", o["bronze_table"])
    if table_exists(b):
        bronze_columns[o["object_id"]] = spark.table(b).columns
print(f"{len(bronze_columns)} of {len(objects)} object(s) have a Bronze table")

ctx = notebookutils.runtime.context
plan = build_mlv_plan(objects, column_map, dq_rules, gold_transforms, bronze_columns=bronze_columns,
                      include_file_bronze=include_file_bronze,
                      workspace_id=ctx.get("currentWorkspaceId", "<WORKSPACE_ID>"),
                      lakehouse_id=ctx.get("defaultLakehouseId", "<LAKEHOUSE_ID>"))
for s in plan["skipped_objects"]:
    print("SKIPPED (no Bronze table yet):", s)
for t in plan["skipped_gold"]:
    print("GOLD NOT GENERATED:", t)
print(f"{len(plan['items'])} materialized lake view(s) planned")

## Turn on Change Data Feed and create the schemas
Without CDF, optimal refresh can only choose between *skip* and *full*. CDF unlocks *incremental*.

In [ ]:
if not dry_run:
    for s in sorted({MLV_SCHEMAS[i["layer"]] for i in plan["items"]}):
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {s}")
    if enable_cdf:
        for stmt in enable_cdf_sql([o for o in objects if o["object_id"] in bronze_columns]):
            spark.sql(stmt)
        if table_exists(tname("meta", "setting")):
            spark.sql(f"ALTER TABLE {tname('meta', 'setting')} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")

## Create the views (upstream first) and record every statement in `meta.mlv_definition`

In [ ]:
results = []
for it in plan["items"]:
    status, err, started = ("PLANNED", None, utcnow())
    if not dry_run:
        try:
            spark.sql(it["ddl"])
            status = "CREATED"
        except Exception as e:
            status, err = "FAILED", f"{type(e).__name__}: {str(e)[:1500]}"
    results.append((it["key"], it["layer"], it["name"], it["object_id"], ", ".join(it["deps"]),
                    "; ".join(it["notes"] + it["warnings"]), it["ddl"], status, err, started))
    print(f"{status:8} {it['name']}" + (f"  <- {err[:200]}" if err else ""))

MLV_DEF_SCHEMA = StructType([StructField(n, t) for n, t in [
    ("mlv_key", StringType()), ("layer", StringType()), ("mlv_name", StringType()), ("source_id", StringType()),
    ("depends_on", StringType()), ("notes", StringType()), ("ddl", StringType()), ("status", StringType()),
    ("error", StringType()), ("generated_at", TimestampType())]])
spark.createDataFrame(results, MLV_DEF_SCHEMA).write.mode("overwrite").option("overwriteSchema", "true") \
     .saveAsTable(tname("meta", "mlv_definition"))
display(spark.table(tname("meta", "mlv_definition")).select("layer", "mlv_name", "depends_on", "status", "notes", "error"))

## Look at what was generated
* `SHOW MATERIALIZED LAKE VIEWS IN silver_mlv` lists the views; `SHOW CREATE MATERIALIZED LAKE VIEW <name>` shows one.
* Lakehouse ribbon > **Materialized lake views > Manage** shows the lineage graph Fabric built from these statements.
* Next: `nb_mdf_55_mlv_refresh` (or a schedule / the pipeline activity) to refresh, then `nb_mdf_56_compare_paths`.

In [ ]:
display(spark.sql(f"SHOW MATERIALIZED LAKE VIEWS IN {MLV_SCHEMAS['silver']}"))
example = next((i for i in plan["items"] if i["layer"] == "silver" and i.get("flag_rules")), plan["items"][0])
print(example["ddl"])

In [ ]:
failed = [r[2] for r in results if r[7] == "FAILED"]
if failed:
    raise Exception(f"{len(failed)} view(s) failed to create: {failed}. Errors are in meta.mlv_definition")
notebookutils.notebook.exit(json.dumps({"created": sum(r[7] == "CREATED" for r in results), "planned": len(results)}))